# AI 辅助学习 - 进阶篇：Flask 图书管理 API 应用

当你告诉 AI：「我想做一个图书管理的 Flask 应用，支持增删改查，用 SQLite 存储」—— AI 帮你搭好框架。

**运行方式**：
```bash
pip install flask
```
然后按顺序执行下方单元格，最后启动服务访问 `http://127.0.0.1:5000/`

> 本教程带你从零搭建一个 RESTful 风格的图书管理 API，涵盖数据库初始化、CRUD 路由和错误处理。

## 1. 导入依赖 & 初始化应用

导入 Flask、SQLite 等模块，创建 Flask 应用实例并指定数据库文件名。

In [ ]:
from flask import Flask, request, jsonify
import sqlite3
import os

app = Flask(__name__)
DB_NAME = "books.db"

## 2. 数据库工具函数

定义两个辅助函数：
- **`get_db()`**：获取数据库连接，并设置 `row_factory` 使结果可以用列名访问
- **`init_db()`**：创建 `books` 表（如果不存在），包含 id、title、author、year、isbn 字段

In [ ]:
def get_db():
    """获取数据库连接"""
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row  # 让查询结果可用列名访问
    return conn


def init_db():
    """初始化数据库表"""
    with get_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS books (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                author TEXT NOT NULL,
                year INTEGER,
                isbn TEXT UNIQUE
            )
        """)
        conn.commit()


# 初始化数据库
init_db()
print("数据库初始化完成")

## 3. API 路由 —— 首页 & 获取图书

定义首页路由（返回 API 使用说明），以及获取所有图书和获取单本图书的 GET 接口。

- `GET /books` — 获取所有图书，支持 `?author=` 参数筛选
- `GET /books/<id>` — 根据 ID 获取单本图书

In [ ]:
@app.route("/")
def home():
    """首页 - API 使用说明"""
    return jsonify({
        "message": "图书管理 API",
        "endpoints": {
            "GET    /books": "获取所有图书",
            "GET    /books/<id>": "获取单本图书",
            "POST   /books": "添加图书 (JSON: title, author, year, isbn)",
            "PUT    /books/<id>": "更新图书 (JSON: 任意字段)",
            "DELETE /books/<id>": "删除图书",
        }
    })


@app.route("/books", methods=["GET"])
def get_books():
    """获取所有图书，支持 ?author= 筛选"""
    author = request.args.get("author")
    with get_db() as conn:
        if author:
            rows = conn.execute(
                "SELECT * FROM books WHERE author LIKE ?",
                (f"%{author}%",)
            ).fetchall()
        else:
            rows = conn.execute("SELECT * FROM books").fetchall()
        return jsonify([dict(r) for r in rows])


@app.route("/books/<int:book_id>", methods=["GET"])
def get_book(book_id):
    """获取单本图书"""
    with get_db() as conn:
        row = conn.execute(
            "SELECT * FROM books WHERE id = ?", (book_id,)
        ).fetchone()
        if row is None:
            return jsonify({"error": "图书不存在"}), 404
        return jsonify(dict(row))

## 4. API 路由 —— 添加图书

`POST /books` 接口接收 JSON 数据来添加新书。
- 必填字段：`title`、`author`
- 可选字段：`year`、`isbn`
- 如果 ISBN 已存在会返回 409 冲突错误

In [ ]:
@app.route("/books", methods=["POST"])
def add_book():
    """添加图书"""
    data = request.get_json()
    required = ["title", "author"]
    for field in required:
        if field not in data:
            return jsonify({"error": f"缺少必填字段: {field}"}), 400

    try:
        with get_db() as conn:
            conn.execute(
                """INSERT INTO books (title, author, year, isbn)
                   VALUES (?, ?, ?, ?)""",
                (data["title"], data["author"],
                 data.get("year"), data.get("isbn"))
            )
            conn.commit()
        return jsonify({"message": "图书添加成功"}), 201
    except sqlite3.IntegrityError:
        return jsonify({"error": "ISBN 已存在"}), 409

## 5. API 路由 —— 更新 & 删除图书

- `PUT /books/<id>` — 更新指定图书的字段（只允许 title、author、year、isbn）
- `DELETE /books/<id>` — 删除指定图书

两个接口都会检查目标图书是否存在，不存在则返回 404。

In [ ]:
@app.route("/books/<int:book_id>", methods=["PUT"])
def update_book(book_id):
    """更新图书"""
    data = request.get_json()
    allowed = ["title", "author", "year", "isbn"]
    updates = {k: v for k, v in data.items() if k in allowed}
    if not updates:
        return jsonify({"error": "没有可更新的字段"}), 400

    set_clause = ", ".join(f"{k} = ?" for k in updates)
    values = list(updates.values()) + [book_id]

    with get_db() as conn:
        cur = conn.execute(
            f"UPDATE books SET {set_clause} WHERE id = ?", values
        )
        conn.commit()
        if cur.rowcount == 0:
            return jsonify({"error": "图书不存在"}), 404
        return jsonify({"message": "图书更新成功"})


@app.route("/books/<int:book_id>", methods=["DELETE"])
def delete_book(book_id):
    """删除图书"""
    with get_db() as conn:
        cur = conn.execute("DELETE FROM books WHERE id = ?", (book_id,))
        conn.commit()
        if cur.rowcount == 0:
            return jsonify({"error": "图书不存在"}), 404
        return jsonify({"message": "图书已删除"})

## 6. 启动服务

运行下方代码启动 Flask 开发服务器。启动后可访问：

- `http://127.0.0.1:5000/` — 查看 API 说明
- `http://127.0.0.1:5000/books` — 获取所有图书列表

> **提示**：在 Jupyter 中运行时，可用 `curl` 或 Postman 等工具测试 API。
> 如需停止服务，点击 Jupyter 的停止按钮或中断内核。

In [ ]:
# 启动 Flask 服务
# 注意：在 notebook 中运行会阻塞后续单元格，建议在终端单独测试
if __name__ == "__main__":
    print("服务启动: http://127.0.0.1:5000")
    app.run(debug=True)

## API 测试示例

你可以在终端中使用以下 `curl` 命令测试各个接口：

```bash
# 添加图书
curl -X POST http://127.0.0.1:5000/books \
  -H "Content-Type: application/json" \
  -d '{"title": "Python编程", "author": "张三", "year": 2024, "isbn": "978-0-123"}'

# 获取所有图书
curl http://127.0.0.1:5000/books

# 按作者筛选
curl http://127.0.0.1:5000/books?author=张三

# 更新图书
curl -X PUT http://127.0.0.1:5000/books/1 \
  -H "Content-Type: application/json" \
  -d '{"year": 2025}'

# 删除图书
curl -X DELETE http://127.0.0.1:5000/books/1
```